# Mesh generation - Rectilinear (DIS + LGR)

Build **structured** (rectilinear) grids over the watershed: a grid with constant row/column spacing, one with variable spacing that refines toward the river, and a local grid refinement (LGR) grid with a nested child grid.

Part of the **mesh-generation** series, adapted from the FloPy watershed geoprocessing example (Hughes and others, 2023, *FloPy Workflows for Creating Structured and Unstructured MODFLOW Models*, Groundwater, https://doi.org/10.1111/gwat.13327). Each notebook builds one family of grids over the same synthetic watershed, samples a fine topographic raster onto the grid, and intersects the river network with the grid cells.

- **Rectilinear (DIS + LGR)** (`mesh-generation-rectilinear`) - structured grids: constant and variable spacing, plus local grid refinement *(this notebook)*
- **Gridgen quadtree** (`mesh-generation-gridgen`) - a quadtree unstructured (DISV) grid built with Gridgen
- **Triangle & Voronoi** (`mesh-generation-triangle-voronoi`) - unstructured (DISV) grids built with Triangle and Voronoi

## Imports and setup

Import FloPy and the shared watershed setup from `mesh_helpers`: the problem extent and contour levels, the fine topographic raster, the boundary/river geometry, and the `resample_topo` / `river_intersection` / `draw_boundary_river` helpers that every mesh-generation notebook uses.

In [ ]:
%matplotlib inline
import flopy
import flopy.plot.styles as styles
import matplotlib.pyplot as plt
import numpy as np
from mesh_helpers import (
    Lx,
    Ly,
    boundary_polygon,
    draw_boundary_river,
    extent,
    levels,
    resample_topo,
    river_intersection,
    vmax,
    vmin,
)
from notebook_helpers import set_idomain

## Constant row and column spacing

A uniform structured grid over the domain; `set_idomain` marks cells inside the watershed boundary active.

In [ ]:
dx = dy = 1666.66666667
nlay = 1
nrow = int(Ly / dy) + 1
ncol = int(Lx / dx) + 1
top = np.ones((nrow, ncol)) * 1000.0
botm = np.ones((nlay, nrow, ncol)) * -100.0
struct_grid = flopy.discretization.StructuredGrid(
    nlay=nlay,
    delr=np.array(ncol * [dx]),
    delc=np.array(nrow * [dy]),
    xoff=0.0,
    yoff=0.0,
    top=top,
    botm=botm,
)
set_idomain(struct_grid, boundary_polygon)

top_sg = resample_topo(struct_grid)
intersection_sg = river_intersection(struct_grid)

In [ ]:
with styles.USGSMap():
    fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
    ax.set_aspect("equal")
    pmv = flopy.plot.PlotMapView(modelgrid=struct_grid, ax=ax)
    pmv.plot_array(top_sg, vmin=vmin, vmax=vmax)
    pmv.plot_array(intersection_sg, masked_values=[0], alpha=0.2, cmap="Reds_r")
    pmv.plot_grid(lw=0.25, color="0.5")
    pmv.contour_array(top_sg, levels=levels, linewidths=0.3, colors="0.75")
    pmv.plot_inactive()
    draw_boundary_river(ax)
    ax.set_title("Constant-spacing structured grid")

**What to look for.** Every cell is the same size, so the river and the fine detail are resolved no better than the rest of the domain. This is the simplest grid, but it spends cells where they are not needed.

## Variable row and column spacing

Refine the grid toward the center by shrinking the cells through a smooth geometric transition, so the fine cells resolve the river while the coarse cells keep the model small.

In [ ]:
# smooth geometric transition between coarse (5000 m) and fine (1666.7 m) cells
multiplier = 1.175
transition = 20000.0
ncells = 7
smoothr = [transition * (multiplier - 1.0) / (multiplier ** float(ncells) - 1.0)]
for i in range(ncells - 1):
    smoothr.append(smoothr[i] * multiplier)
smooth = list(reversed(smoothr))

dx = 9 * [5000] + smooth + 15 * [1666.66666667] + smoothr + 14 * [5000]
dy = 4 * [5000] + smooth + 12 * [1666.66666667] + smoothr + 4 * [5000]
nrow, ncol = len(dy), len(dx)
top = np.ones((nrow, ncol)) * 1000.0
botm = np.ones((1, nrow, ncol)) * -100.0
struct_vrc_grid = flopy.discretization.StructuredGrid(
    nlay=1,
    delr=np.array(dx),
    delc=np.array(dy),
    xoff=0.0,
    yoff=0.0,
    top=top,
    botm=botm,
)
set_idomain(struct_vrc_grid, boundary_polygon)

top_sg_vrc = resample_topo(struct_vrc_grid)
intersection_sg_vrc = river_intersection(struct_vrc_grid)

In [ ]:
with styles.USGSMap():
    fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
    ax.set_aspect("equal")
    pmv = flopy.plot.PlotMapView(modelgrid=struct_vrc_grid, ax=ax)
    pmv.plot_array(top_sg_vrc, vmin=vmin, vmax=vmax)
    pmv.plot_array(intersection_sg_vrc, masked_values=[0], alpha=0.2, cmap="Reds_r")
    pmv.plot_grid(lw=0.25, color="0.5")
    pmv.contour_array(top_sg_vrc, levels=levels, linewidths=0.3, colors="0.75")
    pmv.plot_inactive()
    draw_boundary_river(ax, river_alpha=0.2)
    ax.set_title("Variable-spacing structured grid")

**What to look for.** The cells shrink smoothly toward the middle, so the river is resolved with small cells while the edges stay coarse - fewer cells overall than the constant-spacing grid for similar detail near the stream.

## Local grid refinement (LGR)

Keep a coarse parent grid but cut a hole in it and drop in a finer **child** grid over the area of interest, using `flopy.utils.lgrutil.Lgr` to build the nested child grid.

In [ ]:
from flopy.utils.lgrutil import Lgr

dx = 5000
nrowp, ncolp = int(Ly / dx), int(Lx / dx)
topp = np.ones((nrowp, ncolp)) * 1000.0
botmp = np.ones((1, nrowp, ncolp)) * -100.0
idomainp = np.ones((1, nrowp, ncolp), dtype=int)
idomainp[0, 8:12, 13:18] = 0  # hole where the child grid goes

lgr = Lgr(
    1,
    nrowp,
    ncolp,
    dx,
    dx,
    topp,
    botmp,
    idomainp,
    ncpp=3,
    ncppl=[1],
    xllp=0.0,
    yllp=0.0,
)

struct_gridp = flopy.discretization.StructuredGrid(
    nlay=1,
    delr=np.array(ncolp * [dx]),
    delc=np.array(nrowp * [dx]),
    idomain=idomainp,
    top=topp,
    botm=botmp,
)
set_idomain(struct_gridp, boundary_polygon)

delr_child, delc_child = lgr.get_delr_delc()
xoff, yoff = lgr.get_lower_left()
nrowc, ncolc = delc_child.shape[0], delr_child.shape[0]
struct_gridc = flopy.discretization.StructuredGrid(
    delr=delr_child,
    delc=delc_child,
    xoff=xoff,
    yoff=yoff,
    idomain=idomainp,
    top=np.ones((nrowc, ncolc)) * 1000.0,
    botm=np.ones((1, nrowc, ncolc)) * -100.0,
)
extent_child = struct_gridc.extent

top_ngp = resample_topo(struct_gridp)
top_ngc = resample_topo(struct_gridc)
intersection_ngp = river_intersection(struct_gridp)
intersection_ngp[idomainp[0] == 0] = 0
intersection_ngc = river_intersection(struct_gridc)

In [ ]:
with styles.USGSMap():
    fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
    ax.set_aspect("equal")
    pmv = flopy.plot.PlotMapView(modelgrid=struct_gridp, ax=ax, extent=extent)
    pmv.plot_array(top_ngp, vmin=vmin, vmax=vmax)
    pmv.plot_array(intersection_ngp, masked_values=[0], alpha=0.2, cmap="Reds_r")
    pmv.plot_grid(lw=0.25, color="0.5")
    pmv.contour_array(top_ngp, levels=levels, linewidths=0.3, colors="0.75")
    pmv.plot_inactive(zorder=100)

    pmvc = flopy.plot.PlotMapView(modelgrid=struct_gridc, ax=ax, extent=extent)
    pmvc.plot_array(top_ngc, vmin=vmin, vmax=vmax)
    pmvc.plot_array(intersection_ngc, masked_values=[0], alpha=0.2, cmap="Reds_r")
    pmvc.plot_grid(lw=0.25, color="0.5")
    pmvc.contour_array(top_ngc, levels=levels, linewidths=0.3, colors="0.75")
    draw_boundary_river(ax)
    ax.set_title("Local grid refinement (parent + nested child)")

**What to look for.** A block of the coarse parent grid is replaced by a finer child grid over the area of interest. The two grids meet along the block boundary, where a GWF-GWF exchange transfers flow between them.

**Recap.** Three rectilinear approaches to the same watershed: a uniform grid, a variable-spacing grid that concentrates resolution where it is needed, and an LGR grid with a discrete finer child grid. The next notebooks build *unstructured* grids over the same domain.